# wf2020_near_miss_timing_repair_v1

Purpose: diagnose whether the current best wf_2020 near-miss checkpoint can be timing-repaired into a retained walk-forward candidate, and identify whether the required daily path / actor artifacts exist.

Governance: this is diagnostic only. It does not promote a model or mark an actor retainable.

In [1]:
from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "SPEC.md").exists() and (candidate / "runtime").exists():
            return candidate
    raise FileNotFoundError("could not locate repo root from notebook cwd")

ROOT = find_repo_root(Path.cwd().resolve())
CARD = ROOT / "runtime" / "model_selection" / "card" / "wf2020_near_miss_timing_repair_v1"
FRONTIER = ROOT / "runtime" / "model_selection" / "card" / "fold_frontier_failure_attribution_v1"
CARD.mkdir(parents=True, exist_ok=True)

def rel(path: Path) -> str:
    try:
        return str(path.relative_to(ROOT))
    except ValueError:
        return str(path)

def write_json(path: Path, payload: dict) -> None:
    path.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")

near_path = FRONTIER / "wf2020_near_miss.csv"
frontier_path = FRONTIER / "normalized_candidate_frontier.csv"
near = pd.read_csv(near_path)
frontier = pd.read_csv(frontier_path)
print(f"near_miss_rows={len(near)} frontier_rows={len(frontier)}")

near_miss_rows=14 frontier_rows=68


In [2]:
def source_card_dir(source_card: str) -> Path:
    return ROOT / "runtime" / "model_selection" / "card" / str(source_card)

def artifact_counts(source_card: str) -> dict[str, int]:
    directory = source_card_dir(source_card)
    if not directory.exists():
        return {"actor_files": 0, "daily_paths": 0, "notebooks": 0}
    return {
        "actor_files": len(list(directory.rglob("*.pt"))) + len(list(directory.rglob("*.pth"))),
        "daily_paths": len(list(directory.rglob("*daily_path*.csv"))),
        "notebooks": len(list(directory.rglob("*.ipynb"))),
    }

availability_rows = []
for _, row in near.iterrows():
    source = str(row.get("source_card") or "")
    daily = row.get("daily_path")
    daily_path = ROOT / str(daily) if isinstance(daily, str) and daily.strip() else None
    counts = artifact_counts(source)
    availability_rows.append({
        "source_card": source,
        "fold": row.get("fold"),
        "candidate": row.get("candidate"),
        "candidate_kind": row.get("candidate_kind"),
        "return_pct": row.get("return_pct"),
        "max_dd_pct": row.get("max_dd_pct"),
        "wf2020_return_gap_pp": row.get("wf2020_return_gap_pp"),
        "max_dd_gap_pct": row.get("max_dd_gap_pct"),
        "daily_path": "" if daily_path is None else rel(daily_path),
        "daily_path_exists": bool(daily_path and daily_path.exists()),
        "actor_files_in_source_card": counts["actor_files"],
        "daily_paths_in_source_card": counts["daily_paths"],
        "notebooks_in_source_card": counts["notebooks"],
    })

availability = pd.DataFrame(availability_rows)
availability.to_csv(CARD / "candidate_availability.csv", index=False)
availability.head(10)

,source_card,fold,candidate,candidate_kind,return_pct,max_dd_pct,wf2020_return_gap_pp,max_dd_gap_pct,daily_path,daily_path_exists,actor_files_in_source_card,daily_paths_in_source_card,notebooks_in_source_card
0,cap10_three_fold_checkpoint_metric_replay_v2,wf_2020,ppo_retained_iter_4,ppo_retained,61.324050,20.918285,8.280502,0.918285,,False,0,0,1
1,cap10_three_fold_checkpoint_metric_replay_v2,wf_2020,ppo_selected_best,ppo_selected,61.324050,20.918285,8.280502,0.918285,,False,0,0,1
2,bull_reentry_three_fold_checkpoint_metric_repl...,wf_2020,bc_initialization,bc_initialization,55.597960,15.489488,14.006592,0.000000,,False,0,0,1
3,bull_reentry_three_fold_checkpoint_metric_repl...,wf_2020,ppo_retained_iter_0_bc_equivalent,ppo_retained,55.597960,15.489488,14.006592,0.000000,,False,0,0,1
4,bull_reentry_three_fold_checkpoint_metric_repl...,wf_2020,ppo_retained_iter_4,ppo_retained,50.224828,15.475617,19.379723,0.000000,,False,0,0,1
5,bull_reentry_three_fold_checkpoint_metric_repl...,wf_2020,ppo_selected_best,ppo_selected,50.224828,15.475617,19.379723,0.000000,,False,0,0,1
6,cap10_three_fold_checkpoint_drift_selector_v1,wf_2020,bc_initialization,bc_initialization,46.810017,19.417943,22.794534,0.000000,,False,0,0,1
7,cap10_three_fold_checkpoint_drift_selector_v1,wf_2020,ppo_retained_iter_0_bc_equivalent,ppo_retained,46.810017,19.417943,22.794534,0.000000,,False,0,0,1
8,cap10_three_fold_checkpoint_metric_replay_v2,wf_2020,bc_initialization,bc_initialization,40.580326,14.984701,29.024226,0.000000,,False,0,0,1
9,cap10_three_fold_checkpoint_metric_replay_v2,wf_2020,ppo_retained_iter_0_bc_equivalent,ppo_retained,40.580326,14.984701,29.024226,0.000000,,False,0,0,1


In [3]:
def max_drawdown_pct(nav: pd.Series) -> float:
    nav = pd.to_numeric(nav, errors="coerce").dropna()
    if nav.empty:
        return float("nan")
    drawdown = nav / nav.cummax() - 1.0
    return float(abs(drawdown.min()) * 100.0)

def daily_path_metrics(path: Path) -> dict | None:
    try:
        frame = pd.read_csv(path)
    except Exception:
        return None
    if "strategy_nav" not in frame.columns:
        return None
    date_col = "date" if "date" in frame.columns else "datetime" if "datetime" in frame.columns else None
    if date_col is None:
        return None
    frame[date_col] = pd.to_datetime(frame[date_col])
    frame = frame.sort_values(date_col)
    strategy = pd.to_numeric(frame["strategy_nav"], errors="coerce").dropna()
    if len(strategy) < 2:
        return None
    if "buyhold_nav" in frame.columns:
        buyhold = pd.to_numeric(frame["buyhold_nav"], errors="coerce").dropna()
    else:
        buyhold = pd.Series(dtype=float)
    return_pct = float((strategy.iloc[-1] / strategy.iloc[0] - 1.0) * 100.0)
    buyhold_return_pct = float((buyhold.iloc[-1] / buyhold.iloc[0] - 1.0) * 100.0) if len(buyhold) >= 2 else float("nan")
    avg_position = float(pd.to_numeric(frame.get("position", pd.Series(dtype=float)), errors="coerce").mean()) if "position" in frame.columns else float("nan")
    return {
        "path": rel(path),
        "source_card": path.parent.name,
        "rows": int(len(frame)),
        "start": frame[date_col].iloc[0].strftime("%Y-%m-%d"),
        "end": frame[date_col].iloc[-1].strftime("%Y-%m-%d"),
        "return_pct": return_pct,
        "buyhold_return_pct": buyhold_return_pct,
        "relative_return_pp": return_pct - buyhold_return_pct if np.isfinite(buyhold_return_pct) else float("nan"),
        "max_dd_pct": max_drawdown_pct(strategy),
        "avg_position": avg_position,
    }

paths = sorted((ROOT / "runtime" / "model_selection" / "card").glob("*/*wf_2020*official_daily_path.csv"))
metric_rows = [daily_path_metrics(path) for path in paths]
metric_rows = [row for row in metric_rows if row is not None]
path_metrics = pd.DataFrame(metric_rows)

reference_floor = float(pd.to_numeric(near["wf2020_reference_floor_return_pct"], errors="coerce").dropna().iloc[0])
max_dd_gate = float(pd.to_numeric(near["target_max_dd_pct"], errors="coerce").dropna().iloc[0])
if not path_metrics.empty:
    path_metrics["wf2020_return_gap_pp"] = reference_floor - path_metrics["return_pct"]
    path_metrics["max_dd_gap_pct"] = path_metrics["max_dd_pct"] - max_dd_gate
    path_metrics["wf2020_gate_pass"] = (path_metrics["return_pct"] >= reference_floor) & (path_metrics["max_dd_pct"] < max_dd_gate)
    path_metrics = path_metrics.sort_values(["wf2020_gate_pass", "return_pct"], ascending=[False, False])
path_metrics.to_csv(CARD / "available_daily_path_metrics.csv", index=False)
path_metrics.head(12)

,path,source_card,rows,start,end,return_pct,buyhold_return_pct,relative_return_pp,max_dd_pct,avg_position,wf2020_return_gap_pp,max_dd_gap_pct,wf2020_gate_pass
0,runtime/model_selection/card/return_preserving...,return_preserving_hybrid_target_bc_replay_v1,252,2020-01-02,2020-12-30,14.466331,58.093571,-43.627240,14.831863,0.182782,55.138221,-5.168137,False
6,runtime/model_selection/card/wf2020_riskcap_fu...,wf2020_riskcap_full_bc_replay_v1,252,2020-01-02,2020-12-30,14.057336,58.093571,-44.036235,14.873321,0.169766,55.547216,-5.126679,False
5,runtime/model_selection/card/return_preserving...,return_preserving_hybrid_target_bc_replay_v1,252,2020-01-02,2020-12-30,13.229725,58.093571,-44.863846,34.945908,0.311104,56.374826,14.945908,False
1,runtime/model_selection/card/return_preserving...,return_preserving_hybrid_target_bc_replay_v1,252,2020-01-02,2020-12-30,12.786482,58.093571,-45.307089,14.833673,0.181826,56.818070,-5.166327,False
4,runtime/model_selection/card/return_preserving...,return_preserving_hybrid_target_bc_replay_v1,252,2020-01-02,2020-12-30,5.641055,58.093571,-52.452515,34.945908,0.304803,63.963496,14.945908,False
2,runtime/model_selection/card/return_preserving...,return_preserving_hybrid_target_bc_replay_v1,252,2020-01-02,2020-12-30,-0.151604,58.093571,-58.245175,34.564089,0.283987,69.756156,14.564089,False
3,runtime/model_selection/card/return_preserving...,return_preserving_hybrid_target_bc_replay_v1,252,2020-01-02,2020-12-30,-7.208004,58.093571,-65.301575,34.234895,0.266268,76.812556,14.234895,False


In [4]:
checkpoint_availability = availability[availability["source_card"].astype(str).str.contains("checkpoint", case=False, na=False)]
checkpoint_daily_available = bool(checkpoint_availability["daily_path_exists"].any())
checkpoint_actor_available = bool((checkpoint_availability["actor_files_in_source_card"] > 0).any())
wf2020_top_near_miss = near.sort_values(["return_pct"], ascending=False).iloc[0].to_dict()
best_maxdd_safe = near[near["max_dd_gate"].astype(bool)].sort_values("return_pct", ascending=False).iloc[0].to_dict()

if path_metrics.empty:
    top_available = {}
    pass_count = 0
else:
    top_available = path_metrics.iloc[0].replace({np.nan: None}).to_dict()
    pass_count = int(path_metrics["wf2020_gate_pass"].sum())

summary = {
    "summary": {
        "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "experiment": "wf2020_near_miss_timing_repair_v1",
        "direct_comparison_baseline": "fold_frontier_failure_attribution_v1",
        "source_card": "fold_frontier_failure_attribution_v1",
        "decision": "wf2020_timing_repair_requires_checkpoint_daily_path_export",
        "promotion_allowed": False,
        "actor_retention_allowed": False,
        "training_allowed_from_this_card": False,
        "checkpoint_near_miss_daily_path_available": checkpoint_daily_available,
        "checkpoint_actor_available": checkpoint_actor_available,
        "available_daily_paths_screened": int(len(path_metrics)),
        "available_daily_path_gate_pass_count": pass_count,
        "wf2020_reference_floor_return_pct": reference_floor,
        "target_max_dd_pct": max_dd_gate,
        "highest_checkpoint_near_miss": {
            "source_card": wf2020_top_near_miss.get("source_card"),
            "candidate": wf2020_top_near_miss.get("candidate"),
            "return_pct": float(wf2020_top_near_miss.get("return_pct")),
            "max_dd_pct": float(wf2020_top_near_miss.get("max_dd_pct")),
            "return_gap_pp": float(wf2020_top_near_miss.get("wf2020_return_gap_pp")),
            "max_dd_gap_pct": float(wf2020_top_near_miss.get("max_dd_gap_pct")),
        },
        "best_existing_maxdd_safe_checkpoint": {
            "source_card": best_maxdd_safe.get("source_card"),
            "candidate": best_maxdd_safe.get("candidate"),
            "return_pct": float(best_maxdd_safe.get("return_pct")),
            "max_dd_pct": float(best_maxdd_safe.get("max_dd_pct")),
            "return_gap_pp": float(best_maxdd_safe.get("wf2020_return_gap_pp")),
        },
        "best_available_official_daily_path": top_available,
        "next_mechanism": "export_checkpoint_near_miss_daily_paths_then_rerun_timing_overlay_screen",
        "dashboard_implication": "do_not_offer_interactive_rerun_for_models_without_retained_actor_or_daily_path",
    }
}
write_json(CARD / "summary.json", summary)

selection = {
    "selected": None,
    "experiment": "wf2020_near_miss_timing_repair_v1",
    "direct_comparison_baseline": "fold_frontier_failure_attribution_v1",
    "decision": summary["summary"]["decision"],
    "reason": "The highest wf_2020 checkpoint near-miss has no persisted daily path or actor artifact, so timing repair cannot be validated as a retained model yet.",
    "source_paths": {
        "near_miss": rel(near_path),
        "candidate_availability": rel(CARD / "candidate_availability.csv"),
        "available_daily_path_metrics": rel(CARD / "available_daily_path_metrics.csv"),
    },
}
write_json(CARD / "selection.json", selection)

repair_plan = {
    "role": "diagnose",
    "baseline": "soxl_backtest_policy_reimplementation_v1",
    "direct_comparison": "fold_frontier_failure_attribution_v1",
    "target_window": "wf_2020",
    "accepted_tradeoff": "MaxDD must remain below 20 pct while closing the return gap to the SOXLBacktest buy/sell reference floor.",
    "changed_layer": "selection",
    "external_reference": "soxlbacktest qlib buy/sell baseline",
    "next_steps": [
        "patch checkpoint replay notebooks/scripts to persist fold daily paths for ppo_retained_iter_4 and bc_initialization",
        "rerun fold-aware three-fold replay with daily path export",
        "only then screen timing overlays or reward changes against exported paths",
    ],
}
write_json(CARD / "repair_plan.json", repair_plan)

conclusion = f"""# wf2020_near_miss_timing_repair_v1 Conclusion

Decision: `{summary['summary']['decision']}`.

The highest wf_2020 near-miss is `{summary['summary']['highest_checkpoint_near_miss']['source_card']}:{summary['summary']['highest_checkpoint_near_miss']['candidate']}` with return `{summary['summary']['highest_checkpoint_near_miss']['return_pct']:.2f}%`, MaxDD `{summary['summary']['highest_checkpoint_near_miss']['max_dd_pct']:.2f}%`, return gap `{summary['summary']['highest_checkpoint_near_miss']['return_gap_pp']:.2f}pp`, and MaxDD gap `{summary['summary']['highest_checkpoint_near_miss']['max_dd_gap_pct']:.2f}pp`.

No promotion is allowed from this card because the checkpoint near-miss daily path and actor artifacts are not persisted. The next step is to export the checkpoint daily paths first, then rerun timing overlays against those real paths.

Sidecars: `candidate_availability.csv`, `available_daily_path_metrics.csv`, `summary.json`, `selection.json`, `repair_plan.json`.
"""
(CARD / "conclusion.md").write_text(conclusion, encoding="utf-8")
summary

{'summary': {'generated_at': '2026-06-14T02:09:36+00:00',
  'experiment': 'wf2020_near_miss_timing_repair_v1',
  'direct_comparison_baseline': 'fold_frontier_failure_attribution_v1',
  'source_card': 'fold_frontier_failure_attribution_v1',
  'decision': 'wf2020_timing_repair_requires_checkpoint_daily_path_export',
  'promotion_allowed': False,
  'actor_retention_allowed': False,
  'training_allowed_from_this_card': False,
  'checkpoint_near_miss_daily_path_available': False,
  'checkpoint_actor_available': False,
  'available_daily_paths_screened': 7,
  'available_daily_path_gate_pass_count': 0,
  'wf2020_reference_floor_return_pct': 69.60455156850816,
  'target_max_dd_pct': 20.0,
  'highest_checkpoint_near_miss': {'source_card': 'cap10_three_fold_checkpoint_metric_replay_v2',
   'candidate': 'ppo_retained_iter_4',
   'return_pct': 61.32404958994388,
   'max_dd_pct': 20.918285087095683,
   'return_gap_pp': 8.280501978564274,
   'max_dd_gap_pct': 0.9182850870956828},
  'best_existing_ma

In [5]:
from IPython.display import Markdown, display

conclusion = (CARD / "conclusion.md").read_text(encoding="utf-8")
display(Markdown(conclusion))

# wf2020_near_miss_timing_repair_v1 Conclusion

Decision: `wf2020_timing_repair_requires_checkpoint_daily_path_export`.

The highest wf_2020 near-miss is `cap10_three_fold_checkpoint_metric_replay_v2:ppo_retained_iter_4` with return `61.32%`, MaxDD `20.92%`, return gap `8.28pp`, and MaxDD gap `0.92pp`.

No promotion is allowed from this card because the checkpoint near-miss daily path and actor artifacts are not persisted. The next step is to export the checkpoint daily paths first, then rerun timing overlays against those real paths.

Sidecars: `candidate_availability.csv`, `available_daily_path_metrics.csv`, `summary.json`, `selection.json`, `repair_plan.json`.
